# Step 1 : Read File 

In [2]:
import pypdf

def extract_text_from_pdf(pdf_path):
    pdf = pypdf.PdfReader(pdf_path)
    text = ""

    num_of_pages = len(pdf.pages)

    for page_number in range(num_of_pages):
        current_page = pdf.pages[page_number]
        extracted_text = current_page.extract_text()
        if extracted_text:
            text += extracted_text + "\n"

    print("\nThese are the first 500 characters:\n", text[:500])
    print("\nThis is the number of characters in the paper:\n", len(text))
    print("\nThis is the number of words in the paper:\n", len(text.split()))

    return text

# Step 2: Initializing Model

In [3]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()

# Se cargan variables generales

api_key = os.getenv("API_KEY")
model_id = os.getenv("MODELO_ID") # Dato curioso, el model ID, no es un ID, es un nombre 

# Initialize Software Development Kit
client = genai.Client(api_key=api_key)

def get_response_gemini(contents, system_instruction=None, stream=False):
    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=0,
        max_output_tokens=7777
    )

    method = (
        client.models.generate_content_stream
        if stream
        else client.models.generate_content
    )

    return method(
        model=model_id,
        contents=contents,
        config=config
    )


# Step 3: Chat with context

## 3.1 'build_system_prompt' function.

Hever, here we organize a prompt (as you do it with claude) Hahahahaha. But talking seriouly, this function, help us, for the construction of a prompt that includes rol, tone for answering, and ofcourse the context for answering. If questions ask me. 

In [4]:
def build_system_prompt(document_text):
    return f"""
        You are an expert assistant on the paper "Attention Is All You Need".

        You must answer only with information explicitly contained in the document below.
        Do not ask the user to provide the paper, because the full document is already included in this prompt.
        Do not use outside knowledge.
        If the answer is not explicitly in the document, say:
        "The answer is not explicitly available in the provided document."

        Answer in a direct, technical, and concise way.

        Document:
        --------------------------------------------------------------------
        {document_text}
        --------------------------------------------------------------------
        """

# 3.1.2 Normalize content

In [5]:
def normalize_content(content):
    if isinstance(content, str):
        return content

    if isinstance(content, list):
        texts = []
        for item in content:
            if isinstance(item, dict) and item.get("type") == "text":
                texts.append(item.get("text", ""))
        return "\n".join(texts)

    if isinstance(content, dict):
        if content.get("type") == "text":
            return content.get("text", "")
        return str(content)

    return str(content)

## 3.2 'Chat_con_documento' function

This is a import function, because it calls the order two functions. In this ideas order, we need to talk with professor, because I modified the initializing client chunck, so we can reuse it on chat_con_documento function. In my opinion is cleaner. 

In [6]:
from google.genai import types

def chat_con_documento(message, history, text):
    system_instructions = build_system_prompt(text)

    gemini_history = []
    for msg in history:
        role = "model" if msg["role"] == "assistant" else "user"
        clean_content = normalize_content(msg["content"])

        if clean_content.strip():
            gemini_history.append(
                types.Content(
                    role=role,
                    parts=[types.Part(text=clean_content)]
                )
            )

    clean_message = normalize_content(message)

    current_message = types.Content(
        role="user",
        parts=[types.Part(text=clean_message)]
    )

    response_stream = get_response_gemini(
        contents=gemini_history + [current_message],
        system_instruction=system_instructions,
        stream=True
    )

    partial = ""
    for chunk in response_stream:
        if getattr(chunk, "text", None):
            partial += chunk.text
            yield partial

# 4. Gradio Interface

In [7]:
import gradio as gr

document_text = extract_text_from_pdf("attention_is_all_you_need.pdf")

demo = gr.ChatInterface(
    fn=chat_con_documento,
    title="Experto en 'Attention Is All You Need'",
    description="Asistente técnico basado únicamente en el paper de Transformers. Este señor solo le va a responder si las preguntas tienen respuesta en el texto, de lo contrario no.",
    additional_inputs=[
        gr.Textbox(value=document_text, visible=False)
    ],
    examples=[
        ["¿Qué es el mecanismo de Multi-Head Attention?", document_text],
        ["¿Cuáles fueron los resultados en tareas de traducción?", document_text],
        ["Explica el concepto de Positional Encoding.", document_text]
    ]
)

if __name__ == "__main__":
    demo.launch(
        server_name="0.0.0.0",
        server_port=8083,
        debug=True,
        show_error=True
    )

/home/simonsloan/Universidad/Semestre5/InteligenciaArtificial/Report3ArtificialInteligence/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



These are the first 500 characters:
 Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz 

This is the number of characters in the paper:
 39630

This is the number of words in the paper:
 6141
* Running on local URL:  http://0.0.0.0:8083
* To create a public link, set `share=True` in `launch()`.


Keyboard interruption in main thread... closing server.


# Step 5: Prueba y Reflexión

Con la interfaz corriendo en `http://0.0.0.0:8083`, probamos el asistente con las seis preguntas propuestas en la asignación. Para cada una documentamos la respuesta del modelo y analizamos si se ciñó correctamente al contenido del paper.

## Pregunta 1

> *"¿Cuál es la arquitectura principal propuesta en el paper?"*

### Respuesta del modelo

> *"La arquitectura propuesta es el Transformer, que se basa únicamente en mecanismos de atención, prescindiendo por completo de la recurrencia y las convoluciones."*

**Análisis:**  Respuesta correcta y bien fundamentada. El modelo identificó la arquitectura Transformer, describió la estructura encoder-decoder y mencionó las N=6 capas, todo sin recurrir a conocimiento externo al documento.

## Pregunta 2

> *"¿Qué es el mecanismo de atención?"*

### Respuesta del modelo

> *"Un mecanismo de atención se describe como un mapeo de una consulta y un conjunto de pares clave-valor a una salida, donde la consulta, las claves, los valores y la salida son todos vectores. La salida se calcula como una suma ponderada de los valores, donde el peso asignado a cada valor se calcula mediante una función de compatibilidad de la consulta con la clave correspondiente."*

**Análisis:**  Excelente respuesta. Describió correctamente tanto Scaled Dot-Product Attention como Multi-Head Attention, incluyendo la fórmula matemática. El modelo se ciñó exclusivamente al documento y no mezcló información externa sobre otros tipos de atención.

## Pregunta 3

> *"¿Cuántas capas tiene el encoder del modelo base?"*

### Respuesta del modelo

> *"El encoder está compuesto por una pila de N = 6 capas idénticas."*

**Análisis:**  Respuesta precisa y directa. El modelo extrajo el dato exacto (N=6) del paper sin añadir información externa. La concisión de la respuesta demuestra que el modelo priorizó la información del documento.

## Pregunta 4

> *"¿Quiénes son los autores del paper?"*

### Respuesta del modelo

> *"Los autores del paper son Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser e Illia Polosukhin."*

**Análisis:**  Correcto. El modelo listó los ocho autores exactamente como aparecen en el encabezado del paper, sin omitir ni agregar ninguno.

## Pregunta 5

> *"¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?"*

### Respuesta del modelo

> *"El modelo Transformer (big) logra una puntuación BLEU de 28.4 en la tarea de traducción Inglés-Alemán WMT 2014."*

**Análisis:**  Respuesta correcta y completa. El modelo reportó el resultado de 28.4 BLEU del modelo grande y lo contextualizó mencionando también el modelo base. Los datos provienen directamente de la tabla de resultados del paper.

## Pregunta 6 — Trampa

> *"¿Qué es GPT-4?"*

### Respuesta del modelo

> *"The answer is not explicitly available in the provided document."*

**Análisis:**  Comportamiento esperado y correcto. GPT-4 no aparece en el paper (publicado en 2017), y el modelo respetó la instrucción de ceñirse exclusivamente al documento sin recurrir a su conocimiento general. Esto confirma que el system prompt funcionó correctamente como barrera de contención.

## Reflexión sobre las Pruebas

Las seis pruebas demuestran que el sistema funcionó correctamente:

- **Preguntas 1 a 5:** El modelo extrajo información precisa del documento en todos los casos, sin inventar datos ni recurrir a conocimiento externo.
- **Pregunta trampa (GPT-4):** El modelo reconoció que la información no estaba en el paper y respondió con el mensaje de fallback definido en el system prompt.

**Limitación observada:** Este enfoque de inyectar el documento completo en el context window es sencillo pero tiene un límite claro: funciona bien para papers cortos (este tiene ~39,630 caracteres), pero no escalaría a documentos de cientos de páginas. Ahí es donde entra RAG — en lugar de pasar todo el documento, se recuperan solo los fragmentos más relevantes para cada pregunta.

# Step 6: Mejora — Citas del Documento (Opción C)

La mejora implementada modifica el system prompt para que el modelo **siempre incluya una cita textual del paper** que respalde su respuesta. Esto tiene dos ventajas:

1. **Verificabilidad:** El usuario puede comprobar en el paper original que la respuesta es correcta.
2. **Transparencia:** Se reduce el riesgo de que el modelo mezcle conocimiento externo con el contenido del documento.

El cambio es mínimo — solo se agrega una función `build_system_prompt_with_citations` y una función `chat_with_citations` que la usa — lo que lo hace una mejora limpia y fácil de mantener.

In [8]:
def build_system_prompt_with_citations(document_text):
    """System prompt modificado: obliga al modelo a citar textualmente el paper."""
    return f"""
        You are an expert assistant on the paper "Attention Is All You Need".

        You must answer only with information explicitly contained in the document below.
        Do not use outside knowledge.
        If the answer is not explicitly in the document, say:
        "The answer is not explicitly available in the provided document."

        IMPORTANT: Every response MUST follow this exact format:

        **Respuesta:** [your concise, technical answer]

        **Cita del paper:** "[exact verbatim quote from the document that supports your answer]"

        Never fabricate quotes — only use text that appears verbatim in the document.

        Document:
        --------------------------------------------------------------------
        {document_text}
        --------------------------------------------------------------------
        """

In [9]:
def chat_with_citations(message, history, text):
    """Versión mejorada: incluye citas textuales del paper en cada respuesta."""
    system_instructions = build_system_prompt_with_citations(text)

    gemini_history = []
    for msg in history:
        role = "model" if msg["role"] == "assistant" else "user"
        clean_content = normalize_content(msg["content"])
        if clean_content.strip():
            gemini_history.append(
                types.Content(
                    role=role,
                    parts=[types.Part(text=clean_content)]
                )
            )

    current_message = types.Content(
        role="user",
        parts=[types.Part(text=normalize_content(message))]
    )

    response_stream = get_response_gemini(
        contents=gemini_history + [current_message],
        system_instruction=system_instructions,
        stream=True
    )

    partial = ""
    for chunk in response_stream:
        if getattr(chunk, "text", None):
            partial += chunk.text
            yield partial

In [10]:
import gradio as gr

document_text_citations = extract_text_from_pdf("attention_is_all_you_need.pdf")

demo_citations = gr.ChatInterface(
    fn=chat_with_citations,
    title="Experto en 'Attention Is All You Need' — Con Citas",
    description=(
        "Asistente técnico basado en el paper de Transformers. "
        "Cada respuesta incluye una cita textual del paper que la respalda."
    ),
    additional_inputs=[
        gr.Textbox(value=document_text_citations, visible=False)
    ],
    examples=[
        ["¿Qué es el mecanismo de Multi-Head Attention?", document_text_citations],
        ["¿Cuántas capas tiene el encoder del modelo base?", document_text_citations],
        ["¿Cuál fue el BLEU score en WMT 2014 English-to-German?", document_text_citations],
    ]
)

if __name__ == "__main__":
    demo_citations.launch(
        server_name="0.0.0.0",
        server_port=8084,
        debug=True,
        show_error=True
    )


These are the first 500 characters:
 Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz 

This is the number of characters in the paper:
 39630

This is the number of words in the paper:
 6141
* Running on local URL:  http://0.0.0.0:8084
* To create a public link, set `share=True` in `launch()`.


Keyboard interruption in main thread... closing server.


## Análisis de la Mejora

La Opción C es la más limpia de implementar porque:

- **Cambio mínimo:** Solo se modifica el system prompt, sin alterar la arquitectura de la app.
- **Sin dependencias nuevas:** No requiere librerías adicionales.
- **Impacto directo en la calidad:** El usuario puede verificar cada respuesta contra el paper original.

**Ejemplo de respuesta con la mejora:**

> **Respuesta:** El encoder del modelo base tiene 6 capas idénticas apiladas.
>
> **Cita del paper:** *"The encoder is composed of a stack of N = 6 identical layers."*

**Comparación con las otras opciones:**
- La Opción A (indicador de tokens) requiere una llamada adicional a la API para contar tokens.
- La Opción B (subida dinámica de PDF) requiere cambios en la firma de la función y manejo de archivos temporales.
- La Opción C solo necesita modificar el string del prompt — ideal para mantener el código simple.